In [ ]:
/content/drive/My Drive/FYP/VGGModel/Improved_20250210/current_model_vgg_improved.pth

import torch
import torch.nn as nn
import torch.nn.functional as F

#######################################
# Squeeze-and-Excitation Block
#######################################
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        """
        Squeeze-and-Excitation block: reduces channels by 'reduction', then excites them.
        """
        super(SEBlock, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        # Squeeze
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        # Excite
        return x * y


#######################################
# Manually defined VGG-16 feature extractor
#######################################
def make_vgg16_features():
    """
    This replicates the classic VGG-16 feature layers (convolutions + ReLUs + MaxPools),
    exactly in the same order as torchvision.models.vgg16.features.
    """
    layers = []
    in_channels = 3
    cfg = [64, 64, 'M',
           128, 128, 'M',
           256, 256, 256, 'M',
           512, 512, 512, 'M',
           512, 512, 512, 'M']

    for v in cfg:
        if v == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        else:
            conv2d = nn.Conv2d(in_channels, v, kernel_size=3, padding=1)
            layers += [conv2d, nn.ReLU(inplace=True)]
            in_channels = v
    return nn.Sequential(*layers)


#######################################
# Main VGG-16 Improved Model
#######################################
class VGG16Improved(nn.Module):
    def __init__(self, num_classes=2, freeze_layers=True):
        """
        - num_classes: number of output classes.
        - freeze_layers: if True, freezes the first 24 layers in the feature extractor.
        """
        super(VGG16Improved, self).__init__()

        # 1. Build the VGG-16 convolutional feature extractor manually
        self.features = make_vgg16_features()

        # 2. Optionally freeze early layers (up to index 24 in the Sequential)
        #    This matches `vgg.features[:24]` from the original code.
        if freeze_layers:
            for idx, layer in enumerate(self.features):
                if idx < 24:
                    for param in layer.parameters():
                        param.requires_grad = False

        # 3. Add the Squeeze-and-Excitation attention block after all conv blocks (512 channels at the end)
        self.attention = SEBlock(channel=512, reduction=16)

        # 4. Define a custom classifier.
        #    If you use AdaptiveAvgPool2d(7,7), the feature map shape at the end is (512, 7, 7) = 25088
        #    So the first Linear must map 25088 -> 4096
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        """
        Forward pass:
          - Pass through the VGG-16 features
          - Apply SE attention
          - Adaptive average pooling (7x7)
          - Flatten
          - Pass through the custom classifier
        """
        # Convolution + ReLU + Pooling
        x = self.features(x)

        # Squeeze-and-Excitation
        x = self.attention(x)

        # Force to 7x7 to handle variable input sizes (e.g., 300x300, 224x224, etc.)
        x = F.adaptive_avg_pool2d(x, (7, 7))

        # Flatten
        x = x.view(x.size(0), -1)

        # Classifier
        x = self.classifier(x)
        return x


#######################################
# Example usage
#######################################
if __name__ == "__main__":
    model = VGG16Improved(num_classes=2, freeze_layers=True)

    # Check output shape for a random 300x300 input
    dummy_input = torch.randn(1, 3, 300, 300)
    output = model(dummy_input)
    print("Output shape:", output.shape)  # Should be [1, 2]
